In [1]:
# ============================================================
# DATABASE CONFIGURATION
# ============================================================
# Modify these parameters to connect to your databases

from urllib.parse import quote_plus

# DuckDB Configuration
DUCKDB_FILE = "UrbanPortal.duckdb"

# PostgreSQL Configuration
PG_HOST = "45.79.118.32"
PG_PORT = "5432"
PG_DATABASE = "UrbanPortalDBP"
PG_USER = "postgres"
PG_PASSWORD = "X|($DM$25p%5"
PG_SCHEMA = "urbanportaldbp"

# Attachment name for PostgreSQL in DuckDB
PG_ATTACHMENT_NAME = "urban_postgres"

# Build connection strings with proper URL encoding
POSTGRES_URL = f"postgresql://{PG_USER}:{quote_plus(PG_PASSWORD)}@{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
POSTGRES_URL_SQLALCHEMY = f"postgresql://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DATABASE}"

print(f"Configuration loaded:")
print(f"  DuckDB File: {DUCKDB_FILE}")
print(f"  PostgreSQL: {PG_USER}@{PG_HOST}:{PG_PORT}/{PG_DATABASE}")
print(f"  Schema: {PG_SCHEMA}")
print(f"  Attachment: {PG_ATTACHMENT_NAME}")

Configuration loaded:
  DuckDB File: UrbanPortal.duckdb
  PostgreSQL: postgres@45.79.118.32:5432/UrbanPortalDBP
  Schema: urbanportaldbp
  Attachment: urban_postgres


# Property Data Enrichment Pipeline

## Overview
This notebook enriches property data with comprehensive planning controls, environmental constraints, and land use information by integrating data from PostgreSQL and DuckDB.

## Workflow

```mermaid
graph TD
    A[1. Database Setup] --> B[2. Data Discovery]
    B --> C[3. Layer Configuration]
    C --> D[4. Main Processing]
    D --> E[5. Permissible Uses]
    E --> F[6. PostgreSQL Export]
    
    D --> D1[Property Address Summary]
    D1 --> D2[Lot Geometries]
    D2 --> D3[Spatial Joins]
    D3 --> D4[POI Distances]
    D4 --> D5[Land Valuation]
    D5 --> D6[Process 45+ Planning Layers]
    D6 --> D7[Comprehensive Table]
```

## Key Steps

1. **Database Configuration & Connection**
   - Connect DuckDB to PostgreSQL
   - Load spatial extensions
   - Configure database schemas

2. **Data Discovery** *(Optional Exploration)*
   - Inventory PostgreSQL tables
   - Analyze column structures
   - Validate data sources

3. **Layer Configuration**
   - Define 45+ planning layers (zoning, heritage, environmental, etc.)
   - Configure spatial join parameters
   - Set aggregation methods

4. **Main Processing** (`run_notebook()`)
   - Create property-address mappings
   - Generate lot geometries (centroid + buffered)
   - Calculate POI distances (hospitals, schools, railway stations)
   - Join land valuation data
   - Process all configured layers with spatial joins
   - Build comprehensive property table

5. **Permissible Land Use Enrichment**
   - Split comma-separated zone codes
   - Match with LEP permissible uses
   - Aggregate results per property

6. **Export & Finalize**
   - Export to PostgreSQL
   - Convert geometry columns
   - Create spatial indexes

## Output

Final table: `up_property_comprehensive` with 100+ enriched attributes per property including:
- Planning controls (zoning, FSR, HOB, minimum lot size)
- Environmental constraints (flood, bushfire, biodiversity)
- Heritage overlays
- Proximity to amenities
- Permissible land uses

In [2]:
import duckdb
import pandas as pd
from sqlalchemy import create_engine, text

# Connect to DuckDB
con = duckdb.connect(DUCKDB_FILE) 
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

con.execute("INSTALL postgres_scanner;")
con.execute("LOAD postgres_scanner;")

# Attach PostgreSQL database
con.execute(f"""
    ATTACH '{POSTGRES_URL}' AS {PG_ATTACHMENT_NAME} (TYPE postgres)
""")

# Create SQLAlchemy engine
engine = create_engine(POSTGRES_URL_SQLALCHEMY)

print(f"Connected to DuckDB: {DUCKDB_FILE}")
print(f"Attached PostgreSQL as: {PG_ATTACHMENT_NAME}")

Connected to DuckDB: UrbanPortal.duckdb
Attached PostgreSQL as: urban_postgres


In [3]:
# ============================================================
# POSTGRESQL TABLE AND COLUMN INVENTORY
# ============================================================
# Query all tables and their columns from PostgreSQL schema

table_columns = con.execute(f"""
    SELECT 
        c.table_name,
        c.column_name,
        c.data_type,
        c.is_nullable,
        gc.srid,
        gc.type as geometry_type
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns c
    LEFT JOIN {PG_ATTACHMENT_NAME}.public.geometry_columns gc
        ON c.table_schema = gc.f_table_schema 
        AND c.table_name = gc.f_table_name
        AND c.column_name = gc.f_geometry_column
    WHERE c.table_schema = '{PG_SCHEMA}'
    ORDER BY c.table_name, c.ordinal_position
""").fetchdf()

print(f"Found {table_columns['table_name'].nunique()} tables with {len(table_columns)} total columns")
print(f"\nTables with geometry columns:")
geom_tables = table_columns[table_columns['srid'].notna()][['table_name', 'column_name', 'srid', 'geometry_type']].drop_duplicates()
print(geom_tables.to_string(index=False))
print(f"\n{'='*70}")
table_columns

Found 109 tables with 1866 total columns

Tables with geometry columns:
                                                     table_name           column_name  srid geometry_type
                                                 BCT_Agreements              geometry  4283      GEOMETRY
                                               FloodMapping_z14              geometry  4326      GEOMETRY
                                   Greenfield_Housing_Code_Area              geometry  4326      GEOMETRY
                                                  PNF_Approvals              geometry  4283      GEOMETRY
accelerated_transport_oriented_development_precincts_rezoning_a              geometry  4326      GEOMETRY
                                             acid_sulfate_soils              geometry  4326      GEOMETRY
                                        active_street_frontages              geometry  4326      GEOMETRY
                                                        airport              geo

,table_name,column_name,data_type,is_nullable,srid,geometry_type
0,BCT_Agreements,FID,integer,YES,<NA>,None
1,BCT_Agreements,ITSTITLEST,integer,YES,<NA>,None
2,BCT_Agreements,ITSLOTID,integer,YES,<NA>,None
3,BCT_Agreements,STRATUMLEV,integer,YES,<NA>,None
4,BCT_Agreements,HASSTRATUM,integer,YES,<NA>,None
...,...,...,...,...,...,...
1861,wetlands,published_date,bigint,YES,<NA>,None
1862,wetlands,commenced_date,bigint,YES,<NA>,None
1863,wetlands,currency_date,bigint,YES,<NA>,None
1864,wetlands,amendment,character varying,YES,<NA>,None


In [4]:
# Check specific table columns to understand naming conventions
print("Checking column naming conventions...\n")

# Check Suburbs table
suburbs_cols = con.execute(f"""
    SELECT column_name 
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
    WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'Suburbs'
    ORDER BY ordinal_position
""").fetchdf()
print("Suburbs columns:", suburbs_cols['column_name'].tolist()[:10])

# Check Property table
property_cols = con.execute(f"""
    SELECT column_name 
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
    WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'Property'
    ORDER BY ordinal_position
""").fetchdf()
print("Property columns:", property_cols['column_name'].tolist()[:10])

# Check addressstring table
address_cols = con.execute(f"""
    SELECT column_name 
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
    WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'addressstring'
    ORDER BY ordinal_position
""").fetchdf()
print("addressstring columns:", address_cols['column_name'].tolist()[:10])

# Check Land_Zoning_Map columns
lzn_cols = con.execute(f"""
    SELECT column_name 
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
    WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'Land_Zoning_Map'
    ORDER BY ordinal_position
""").fetchdf()
print("Land_Zoning_Map columns:", lzn_cols['column_name'].tolist()[:10])

# Check Lot table
lot_cols = con.execute(f"""
    SELECT column_name 
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
    WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'Lot'
    ORDER BY ordinal_position
""").fetchdf()
print("Lot columns:", lot_cols['column_name'].tolist()[:10])

Checking column naming conventions...

Suburbs columns: []
Property columns: []
addressstring columns: ['id', 'containment', 'contributoralignment', 'contributorid', 'contributororigin', 'council', 'createdate', 'deliverypointbarcode', 'deliverypointid', 'enddate']
Land_Zoning_Map columns: []
Lot columns: []


In [5]:
# Get full column lists for key tables
print("\n=== FULL COLUMN LISTS ===\n")

print("All Suburbs columns:")
print(suburbs_cols['column_name'].tolist())

print("\n\nAll addressstring columns:")
print(address_cols['column_name'].tolist())


=== FULL COLUMN LISTS ===

All Suburbs columns:
[]


All addressstring columns:
['id', 'containment', 'contributoralignment', 'contributorid', 'contributororigin', 'council', 'createdate', 'deliverypointbarcode', 'deliverypointid', 'enddate', 'gnafprimarysiteid', 'objectid', 'gurasid', 'housenumber', 'housenumberfirst', 'housenumberfirstprefix', 'housenumberfirstsuffix', 'housenumbersecond', 'housenumbersecondprefix', 'housenumbersecondsuffix', 'lastupdate', 'lastupdatestring', 'address', 'levelnumber', 'levelnumberprefix', 'levelnumbersuffix', 'leveltype', 'locationdescription', 'msoid', 'officialaddressstringoid', 'postcode', 'principaladdresssiteoid', 'principaladdresstype', 'addressconfidence', 'privatestreetname', 'privatestreetsuffix', 'privatestreettype', 'processstate', 'propid', 'roadname', 'roadside', 'roadsuffix', 'roadtype', 'routeoid', 'addresssitename', 'ruraladdress', 'secondroadname', 'secondroadsuffix', 'secondroadtype', 'sppropid', 'startdate', 'state', 'suburbname',

In [6]:
# ============================================================
# SEARCH FOR COASTAL AND RELATED TABLES
# ============================================================
# Search for tables containing coastal, wetland, drinking, riparian, mine, scenic keywords

search_keywords = ['coastal', 'wetland', 'drinking', 'riparian', 'mine', 'scenic']

print(f"\n{'='*70}")
print(f"SEARCHING FOR COASTAL AND ENVIRONMENTAL TABLES")
print(f"{'='*70}\n")

for keyword in search_keywords:
    print(f"\n🔍 Tables containing '{keyword}':")
    tables = con.execute(f"""
        SELECT DISTINCT 
            c.table_name,
            gc.type as geometry_type,
            gc.srid
        FROM {PG_ATTACHMENT_NAME}.information_schema.columns c
        LEFT JOIN {PG_ATTACHMENT_NAME}.public.geometry_columns gc
            ON c.table_schema = gc.f_table_schema 
            AND c.table_name = gc.f_table_name
            AND c.column_name = gc.f_geometry_column
        WHERE c.table_schema = '{PG_SCHEMA}' 
            AND LOWER(c.table_name) LIKE '%{keyword}%'
        ORDER BY c.table_name
    """).fetchdf()
    
    if len(tables) > 0:
        for _, row in tables.iterrows():
            print(f"   ✓ {row['table_name']}")
            if pd.notna(row['geometry_type']):
                print(f"     Geometry: {row['geometry_type']} (SRID: {row['srid']})")
            
            # Get columns for this table
            cols = con.execute(f"""
                SELECT column_name 
                FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
                WHERE table_schema = '{PG_SCHEMA}' AND table_name = '{row['table_name']}'
                ORDER BY ordinal_position
            """).fetchdf()
            
            print(f"     Columns: {', '.join(cols['column_name'].tolist())}")
            print()
    else:
        print(f"   ✗ No tables found")

print(f"\n{'='*70}")


SEARCHING FOR COASTAL AND ENVIRONMENTAL TABLES


🔍 Tables containing 'coastal':
   ✓ coastal_environment_area_map
     Geometry: POLYGON (SRID: 4326)
     Columns: ogc_fid, epi_type, geometry, objectid, epi_name, published_date, commenced_date, currency_date, amendment, map_name, label

   ✓ coastal_environment_area_map
     Columns: ogc_fid, epi_type, geometry, objectid, epi_name, published_date, commenced_date, currency_date, amendment, map_name, label

   ✓ coastal_use_area_map
     Columns: ogc_fid, epi_type, geometry, objectid, epi_name, published_date, commenced_date, currency_date, amendment, map_name, label

   ✓ coastal_use_area_map
     Geometry: POLYGON (SRID: 4326)
     Columns: ogc_fid, epi_type, geometry, objectid, epi_name, published_date, commenced_date, currency_date, amendment, map_name, label

   ✓ coastal_wetlands
     Geometry: POLYGON (SRID: 4326)
     Columns: ogc_fid, epi_type, geometry, objectid, epi_name, published_date, commenced_date, currency_date, amendme

In [7]:
# ============================================================
# VERIFY PROBLEMATIC TABLES
# ============================================================
# Check for: Lot_Size_Map, Active_Street_Frontages, Drinking_Water_Catchment, 
# Scenic_Protection_Land, Heritage_Conservation_Area

problematic_keywords = ['lot.*size', 'active.*street', 'heritage.*conservation']

print(f"\n{'='*70}")
print(f"VERIFYING PROBLEMATIC TABLES")
print(f"{'='*70}\n")

for keyword in problematic_keywords:
    print(f"\n🔍 Tables matching '{keyword}':")
    tables = con.execute(f"""
        SELECT DISTINCT 
            c.table_name,
            gc.type as geometry_type
        FROM {PG_ATTACHMENT_NAME}.information_schema.columns c
        LEFT JOIN {PG_ATTACHMENT_NAME}.public.geometry_columns gc
            ON c.table_schema = gc.f_table_schema 
            AND c.table_name = gc.f_table_name
            AND c.column_name = gc.f_geometry_column
        WHERE c.table_schema = '{PG_SCHEMA}' 
            AND LOWER(c.table_name) LIKE '%{keyword.replace(".*", "%")}%'
        ORDER BY c.table_name
    """).fetchdf()
    
    if len(tables) > 0:
        for _, row in tables.iterrows():
            print(f"   ✓ {row['table_name']}")
            
            # Get columns for this table
            cols = con.execute(f"""
                SELECT column_name 
                FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
                WHERE table_schema = '{PG_SCHEMA}' AND table_name = '{row['table_name']}'
                ORDER BY ordinal_position
            """).fetchdf()
            
            print(f"     Columns: {', '.join(cols['column_name'].tolist())}")
            print()
    else:
        print(f"   ✗ No tables found")

# Directly check the specific tables that are failing
print(f"\n{'='*70}")
print(f"DIRECT TABLE CHECKS")
print(f"{'='*70}\n")

specific_tables = [
    'Active_Street_Frontages',
    'Drinking_Water_Catchment', 
    'Scenic_Protection_Land'
]

for table_name in specific_tables:
    print(f"\n📊 {table_name}:")
    try:
        cols = con.execute(f"""
            SELECT column_name 
            FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
            WHERE table_schema = '{PG_SCHEMA}' AND table_name = '{table_name}'
            ORDER BY ordinal_position
        """).fetchdf()
        
        if len(cols) > 0:
            print(f"   ✓ Table exists")
            print(f"   Columns: {', '.join(cols['column_name'].tolist())}")
        else:
            print(f"   ✗ Table not found")
    except Exception as e:
        print(f"   ✗ Error: {e}")

print(f"\n{'='*70}")


VERIFYING PROBLEMATIC TABLES


🔍 Tables matching 'lot.*size':
   ✓ minimum_lot_size
     Columns: ogc_fid, sym_code, lot_size, units, legis_ref_area, legis_ref_clause, epi_type, geometry, objectid, epi_name, lga_name, published_date, commenced_date, currency_date, amendment, lay_class

   ✓ minimum_lot_size
     Columns: ogc_fid, sym_code, lot_size, units, legis_ref_area, legis_ref_clause, epi_type, geometry, objectid, epi_name, lga_name, published_date, commenced_date, currency_date, amendment, lay_class


🔍 Tables matching 'active.*street':
   ✓ active_street_frontages
     Columns: ogc_fid, epi_type, geometry, objectid, epi_name, lga_name, published_date, commenced_date, currency_date, amendment, lay_class

   ✓ active_street_frontages
     Columns: ogc_fid, epi_type, geometry, objectid, epi_name, lga_name, published_date, commenced_date, currency_date, amendment, lay_class


🔍 Tables matching 'heritage.*conservation':
   ✗ No tables found

DIRECT TABLE CHECKS


📊 Active_Street_Fro

In [8]:
# Search for heritage conservation tables
print(f"\n{'='*70}")
print(f"SEARCHING FOR HERITAGE TABLES")
print(f"{'='*70}\n")

heritage_tables = con.execute(f"""
    SELECT DISTINCT c.table_name
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns c
    WHERE c.table_schema = '{PG_SCHEMA}' 
        AND LOWER(c.table_name) LIKE '%heritage%'
    ORDER BY c.table_name
""").fetchdf()

for _, row in heritage_tables.iterrows():
    print(f"\n✓ {row['table_name']}")
    
    # Get columns
    cols = con.execute(f"""
        SELECT column_name 
        FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
        WHERE table_schema = '{PG_SCHEMA}' AND table_name = '{row['table_name']}'
        ORDER BY ordinal_position
    """).fetchdf()
    
    print(f"  Columns: {', '.join(cols['column_name'].tolist())}")


SEARCHING FOR HERITAGE TABLES


✓ epi_heritage
  Columns: ogc_fid, h_id, h_name, sig, legis_ref_clause, epi_type, geometry, objectid, epi_name, lga_name, published_date, commenced_date, currency_date, amendment, lay_class

✓ heritage
  Columns: ogc_fid, map_name, lay_name, lay_class, h_id, h_name, sig, legis_ref_clause, pco_ref_key, epi_type, geometry, objectid, epi_name, lga_name, published_date, commenced_date, currency_date, amendment, map_type


In [9]:
# Check columns for HIGH PRIORITY PLANNING CONTROL tables
print("\n=== HIGH PRIORITY PLANNING TABLES ===\n")

# FSR
fsr_cols = con.execute(f"""
    SELECT column_name 
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
    WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'Floor_Space_Ratio_Map'
    ORDER BY ordinal_position
""").fetchdf()
print("Floor_Space_Ratio_Map columns:", fsr_cols['column_name'].tolist())

# HOB
hob_cols = con.execute(f"""
    SELECT column_name 
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
    WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'Height_of_Buildings_Map'
    ORDER BY ordinal_position
""").fetchdf()
print("\nHeight_of_Buildings_Map columns:", hob_cols['column_name'].tolist())

# Minimum Lot Size
mls_cols = con.execute(f"""
    SELECT column_name 
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
    WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'Minimum_Lot_Size'
    ORDER BY ordinal_position
""").fetchdf()
print("\nMinimum_Lot_Size columns:", mls_cols['column_name'].tolist())

# LEP
lep_cols = con.execute(f"""
    SELECT column_name 
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
    WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'Local_Environmental_Plan'
    ORDER BY ordinal_position
""").fetchdf()
print("\nLocal_Environmental_Plan columns:", lep_cols['column_name'].tolist())

# DCP
dcp_cols = con.execute(f"""
    SELECT column_name 
    FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
    WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'Development_Control_Plan_LGA_Based'
    ORDER BY ordinal_position
""").fetchdf()
print("\nDevelopment_Control_Plan_LGA_Based columns:", dcp_cols['column_name'].tolist())


=== HIGH PRIORITY PLANNING TABLES ===

Floor_Space_Ratio_Map columns: []

Height_of_Buildings_Map columns: []

Minimum_Lot_Size columns: []

Local_Environmental_Plan columns: []

Development_Control_Plan_LGA_Based columns: []


In [10]:
# ============================================================
# ENVIRONMENTAL LAYERS - Column Discovery
# ============================================================
# Query columns for each environmental layer to determine what to include

environmental_tables = [
    'Biodiversity_Map',
    'Biodiversity_Values',
    'Environmentally_Sensitive_Land',
    'Identified_Wilderness',
    'Mineral_and_Resource_Land',
    'Natural_Resources_Biodiversity_Map',
    'Natural_Resources_Sensitivity_Map',
    'Natural_Resources_Water_Map',
    'Salinity'
]

print(f"\n{'='*70}")
print(f"ENVIRONMENTAL LAYERS - COLUMN ANALYSIS")
print(f"{'='*70}\n")

for table in environmental_tables:
    cols = con.execute(f"""
        SELECT column_name, data_type
        FROM {PG_ATTACHMENT_NAME}.information_schema.columns 
        WHERE table_schema = '{PG_SCHEMA}' AND table_name = '{table}'
        ORDER BY ordinal_position
    """).fetchdf()
    
    print(f"📊 {table}")
    print(f"   Columns: {', '.join(cols['column_name'].tolist())}")
    
    # Check for common useful columns
    useful_cols = [col for col in cols['column_name'].tolist() 
                   if any(keyword in col.upper() for keyword in 
                         ['LAY_CLASS', 'LAY_NAME', 'LABEL', 'CLASS', 'TYPE', 'CATEGORY', 'NAME', 'DESCRIPTION', 'VALUE'])]
    
    if useful_cols:
        print(f"   ✓ Suggested columns: {', '.join(useful_cols)}")
    print()



ENVIRONMENTAL LAYERS - COLUMN ANALYSIS

📊 Biodiversity_Map
   Columns: 

📊 Biodiversity_Values
   Columns: 

📊 Environmentally_Sensitive_Land
   Columns: 

📊 Identified_Wilderness
   Columns: 

📊 Mineral_and_Resource_Land
   Columns: 

📊 Natural_Resources_Biodiversity_Map
   Columns: 

📊 Natural_Resources_Sensitivity_Map
   Columns: 

📊 Natural_Resources_Water_Map
   Columns: 

📊 Salinity
   Columns: 



# Layer Configuration

This section defines all planning layers to be processed. Each layer configuration includes:
- **name**: Layer identifier used in table names
- **source_table**: PostgreSQL source table name
- **join_geom**: Geometry to use for spatial joins ('centroid_geom' or 'buffered_geom')
- **columns**: Dictionary mapping source columns to target column names
- **where_clause**: SQL WHERE condition to filter results
- **table_alias**: SQL table alias

Special parameters:
- **join_type**: Use 'address' or 'propid' instead of geometry join
- **include_address**: Add address column to output
- **include_propid**: Add propid column to output
- **aggregation_method**: 'max_and_string' for mixed numeric/text columns

In [11]:
# ============================================================
# LAYER CONFIGURATION
# ============================================================
# Define all layers to be processed with their attributes
# Updated for urbanportaldbp schema - uses PascalCase table names and UPPERCASE columns

layer_config = [
    
    # ============================================================
    # BASE / REFERENCE LAYERS
    # ============================================================
    {
        'name': 'suburb',
        'source_table': 'Suburbs',
        'join_geom': 'centroid_geom',
        'columns': {'SUBURBNAME': 'suburbname', 'POSTCODE': 'postcode'},
        'where_clause': 's.SUBURBNAME IS NOT NULL',
        'table_alias': 's'
    },
    {
        'name': 'lga',
        'source_table': 'Local_Government_Area',
        'join_geom': 'centroid_geom',
        'columns': {'LGANAME': 'lga_name', 'COUNCILNAME': 'council_name'},
        'where_clause': 'lga.LGANAME IS NOT NULL',
        'table_alias': 'lga'
    },
    
    # ============================================================
    # PLANNING CONTROLS
    # ============================================================
    {
        'name': 'lep',
        'source_table': 'Local_Environmental_Plan',
        'join_geom': 'buffered_geom',
        'columns': {'EPI_NAME': 'lep_name', 'LGA_NAME': 'lep_lga_name', 'CURRENCY_DATE': 'lep_currency_date', 'LAY_CLASS': 'lep_lay_class'},
        'where_clause': 'lep.EPI_NAME IS NOT NULL',
        'table_alias': 'lep'
    },
    {
        'name': 'dcp',
        'source_table': 'Development_Control_Plan_LGA_Based',
        'join_geom': 'buffered_geom',
        'columns': {'PLAN_NAME': 'dcp_plan_name', 'LGA_NAME': 'dcp_lga_name', 'COUNCIL_NAME': 'dcp_council_name', 'PLAN_TYPE': 'dcp_plan_type'},
        'where_clause': 'dcp.PLAN_NAME IS NOT NULL',
        'table_alias': 'dcp'
    },
    {
        'name': 'lzn',
        'source_table': 'Land_Zoning_Map',
        'join_geom': 'buffered_geom',
        'columns': {'SYM_CODE': 'lzn_sym_code', 'LAY_CLASS': 'lzn_lay_class', 'EPI_NAME': 'lzn_epi_name', 'LABEL': 'lzn_label'},
        'where_clause': 'lzn.SYM_CODE IS NOT NULL AND lzn.LAY_CLASS IS NOT NULL',
        'table_alias': 'lzn',
        'include_address': True,
        'include_propid': True
    },
    {
        'name': 'lzn_historic',
        'source_table': 'land_zoning_map_historic',
        'join_geom': 'buffered_geom',
        'columns': {
            'sym_code': 'historic_zone',
            'lay_class': 'historic_lay_class',
            'amendment': 'historic_amendment',
            'commenced_datetime': 'historic_commenced_date',
            'published_datetime': 'historic_published_date'
        },
        'where_clause': "lznh.amendment IS NOT NULL AND lznh.published_datetime::VARCHAR NOT LIKE '9999%'",
        'table_alias': 'lznh',
        'aggregation_method': 'string_agg'
    },
    {
        'name': 'fsr',
        'source_table': 'Floor_Space_Ratio_Map',
        'join_geom': 'buffered_geom',
        'columns': {'SYM_CODE': 'fsr_sym_code', 'LAY_CLASS': 'fsr_lay_class', 'LABEL': 'fsr_label', 'FSR': 'fsr_value', 'EPI_NAME': 'fsr_epi_name'},
        'where_clause': 'fsr.FSR IS NOT NULL',
        'table_alias': 'fsr'
    },
    {
        'name': 'hob',
        'source_table': 'Height_of_Buildings_Map',
        'join_geom': 'buffered_geom',
        'columns': {'SYM_CODE': 'hob_sym_code', 'MAX_B_H': 'hob_max_height', 'UNITS': 'hob_units', 'MAX_B_H_M': 'hob_max_height_m', 'EPI_NAME': 'hob_epi_name'},
        'where_clause': 'hob.MAX_B_H IS NOT NULL',
        'table_alias': 'hob'
    },
    {
        'name': 'minlotsize',
        'source_table': 'Minimum_Lot_Size',
        'join_geom': 'buffered_geom',
        'columns': {'SYM_CODE': 'mls_sym_code', 'LAY_CLASS': 'mls_lay_class', 'LOT_SIZE': 'mls_lot_size', 'UNITS': 'mls_units', 'EPI_NAME': 'mls_epi_name'},
        'where_clause': 'mls.LOT_SIZE IS NOT NULL',
        'table_alias': 'mls'
    },
    {
        'name': 'activestreetfront',
        'source_table': 'Active_Street_Frontages',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'asf_lay_class', 'EPI_NAME': 'asf_epi_name', 'LGA_NAME': 'asf_lga_name'},
        'where_clause': 'asf.LAY_CLASS IS NOT NULL',
        'table_alias': 'asf'
    },
    {
        'name': 'lmr',
        'source_table': 'lmr',
        'join_geom': 'buffered_geom',
        'columns': {
            'buffer': 'buffer',
            'lmr_permissible': 'lmr_permissible',
            'lmr_height_rfb': 'lmr_height_rfb',
            'lmr_height_sth': 'lmr_height_sth'
        },
        'where_clause': 'lmr.geom IS NOT NULL',
        'table_alias': 'lmr'
    },
    {
        'name': 'foreshorebuildline',
        'source_table': 'Foreshore_Building_Line',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'fbl_lay_class', 'EPI_NAME': 'fbl_epi_name', 'LGA_NAME': 'fbl_lga_name'},
        'where_clause': 'fbl.LAY_CLASS IS NOT NULL',
        'table_alias': 'fbl'
    },
    {
        'name': 'greenfieldhousecode',
        'source_table': 'Greenfield_Housing_Code_Area',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'ghc_lay_class'},
        'where_clause': 'ghc.LAY_CLASS IS NOT NULL',
        'table_alias': 'ghc'
    },
    
    # ============================================================
    # ENVIRONMENTAL CONSTRAINTS
    # ============================================================
    {
        'name': 'flood',
        'source_table': 'Flood_Planning_Map',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'floodmapping'},
        'where_clause': 'fp.LAY_CLASS IS NOT NULL',
        'table_alias': 'fp'
    },
    {
        'name': 'bushfire',
        'source_table': 'Bushfire_Prone_Land',
        'join_geom': 'buffered_geom',
        'columns': {'d_Category': 'bushfireproneland'},
        'where_clause': 'bpl.d_Category IS NOT NULL',
        'table_alias': 'bpl'
    },
    {
        'name': 'acidsulfatesoils',
        'source_table': 'Acid_Sulfate_Soils',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'ass_lay_class'},
        'where_clause': 'ass.LAY_CLASS IS NOT NULL',
        'table_alias': 'ass'
    },
    {
        'name': 'biodiversity',
        'source_table': 'Terrestrial_Biodiversity',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'biodiversity'},
        'where_clause': 'tb.LAY_CLASS IS NOT NULL',
        'table_alias': 'tb'
    },
    {
        'name': 'groundwater',
        'source_table': 'Groundwater_Vulnerability',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'groundwatervulnerability'},
        'where_clause': 'gv.LAY_CLASS IS NOT NULL',
        'table_alias': 'gv'
    },
    {
        'name': 'landslide',
        'source_table': 'Landslide_Risk_Land',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'landsliderisk'},
        'where_clause': 'lsr.LAY_CLASS IS NOT NULL',
        'table_alias': 'lsr'
    },
    {
        'name': 'biodiversitymap',
        'source_table': 'Biodiversity_Map',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'biomap_lay_class', 'LAY_NAME': 'biomap_lay_name', 'LABEL': 'biomap_label', 'EPI_NAME': 'biomap_epi_name'},
        'where_clause': 'bm.LAY_CLASS IS NOT NULL',
        'table_alias': 'bm'
    },
    {
        'name': 'biodiversityvalues',
        'source_table': 'Biodiversity_Values',
        'join_geom': 'buffered_geom',
        'columns': {'BOSET_Class': 'biovalue_boset_class', 'BV_Category': 'biovalue_category'},
        'where_clause': 'bv.BV_Category IS NOT NULL',
        'table_alias': 'bv'
    },
    {
        'name': 'envsensitiveland',
        'source_table': 'Environmentally_Sensitive_Land',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'envsensi_lay_class', 'EPI_NAME': 'envsensi_epi_name', 'LGA_NAME': 'envsensi_lga_name'},
        'where_clause': 'esl.LAY_CLASS IS NOT NULL',
        'table_alias': 'esl'
    },
    {
        'name': 'wilderness',
        'source_table': 'Identified_Wilderness',
        'join_geom': 'buffered_geom',
        'columns': {'NAME': 'wilderness_name'},
        'where_clause': 'iw.NAME IS NOT NULL',
        'table_alias': 'iw'
    },
    {
        'name': 'mineralresource',
        'source_table': 'Mineral_and_Resource_Land',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'mineral_lay_class', 'EPI_NAME': 'mineral_epi_name', 'EPI_TYPE': 'mineral_epi_type'},
        'where_clause': 'mrl.LAY_CLASS IS NOT NULL',
        'table_alias': 'mrl'
    },
    {
        'name': 'natresbiodiversity',
        'source_table': 'Natural_Resources_Biodiversity_Map',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'nrbio_lay_class', 'EPI_NAME': 'nrbio_epi_name', 'EPI_TYPE': 'nrbio_epi_type'},
        'where_clause': 'nrbm.LAY_CLASS IS NOT NULL',
        'table_alias': 'nrbm'
    },
    {
        'name': 'natressensitivity',
        'source_table': 'Natural_Resources_Sensitivity_Map',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'nrsensi_lay_class', 'LAY_NAME': 'nrsensi_lay_name', 'EPI_NAME': 'nrsensi_epi_name'},
        'where_clause': 'nrsm.LAY_CLASS IS NOT NULL',
        'table_alias': 'nrsm'
    },
    {
        'name': 'natreswater',
        'source_table': 'Natural_Resources_Water_Map',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'nrwater_lay_class', 'LAY_NAME': 'nrwater_lay_name', 'EPI_NAME': 'nrwater_epi_name'},
        'where_clause': 'nrwm.LAY_CLASS IS NOT NULL',
        'table_alias': 'nrwm'
    },
    {
        'name': 'salinity',
        'source_table': 'Salinity',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'salinity_lay_class', 'EPI_NAME': 'salinity_epi_name'},
        'where_clause': 'sal.LAY_CLASS IS NOT NULL',
        'table_alias': 'sal'
    },
    {
        'name': 'koalahabitat',
        'source_table': 'koala_habitat_map',
        'join_geom': 'buffered_geom',
        'columns': {'lay_name': 'koala_lay_name', 'lay_class': 'koala_lay_class'},
        'where_clause': 'koala.lay_class IS NOT NULL',
        'table_alias': 'koala'
    },
    {
        'name': 'coastalwetlands',
        'source_table': 'Coastal_Wetlands',
        'join_geom': 'buffered_geom',
        'columns': {'LABEL': 'coastal_wetlands', 'MAP_NAME': 'cwet_map_name', 'EPI_NAME': 'cwet_epi_name'},
        'where_clause': 'cwet.LABEL IS NOT NULL',
        'table_alias': 'cwet'
    },
    {
        'name': 'coastalenvironment',
        'source_table': 'Coastal_Environment_Area_Map',
        'join_geom': 'buffered_geom',
        'columns': {'LABEL': 'coastal_environment_area', 'MAP_NAME': 'cenv_map_name', 'EPI_NAME': 'cenv_epi_name'},
        'where_clause': 'cenv.LABEL IS NOT NULL',
        'table_alias': 'cenv'
    },
    {
        'name': 'coastaluse',
        'source_table': 'Coastal_Use_Area_Map',
        'join_geom': 'buffered_geom',
        'columns': {'LABEL': 'coastal_use_area', 'MAP_NAME': 'cuse_map_name', 'EPI_NAME': 'cuse_epi_name'},
        'where_clause': 'cuse.LABEL IS NOT NULL',
        'table_alias': 'cuse'
    },
    {
        'name': 'drinkingwatercatch',
        'source_table': 'Drinking_Water_Catchment',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'drinking_water_catchment', 'LGA_NAME': 'dwc_lga_name', 'EPI_NAME': 'dwc_epi_name'},
        'where_clause': 'dwc.LAY_CLASS IS NOT NULL',
        'table_alias': 'dwc'
    },
    {
        'name': 'riparianland',
        'source_table': 'Riparian_Land_and_Watercourses',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'riparianlandwatercourse', 'LAY_NAME': 'rip_lay_name', 'EPI_NAME': 'rip_epi_name'},
        'where_clause': 'rip.LAY_CLASS IS NOT NULL',
        'table_alias': 'rip'
    },
    {
        'name': 'minesubsidence',
        'source_table': 'Mine_Subsidence_District',
        'join_geom': 'buffered_geom',
        'columns': {'DISTRICTNAME': 'mine_subsidence_district'},
        'where_clause': 'msub.DISTRICTNAME IS NOT NULL',
        'table_alias': 'msub'
    },
    {
        'name': 'scenicprotection',
        'source_table': 'Scenic_Protection_Land',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'scenicprotectionland', 'LGA_NAME': 'scenic_lga_name', 'EPI_NAME': 'scenic_epi_name'},
        'where_clause': 'scenic.LAY_CLASS IS NOT NULL',
        'table_alias': 'scenic'
    },
    
    # ============================================================
    # HERITAGE & CULTURAL
    # ============================================================
    {
        'name': 'heritage',
        'source_table': 'EPI_Heritage',
        'join_geom': 'buffered_geom',
        'columns': {'H_NAME': 'h_name', 'H_ID': 'h_id', 'LAY_CLASS': 'heritage_class'},
        'where_clause': 'eh.H_NAME IS NOT NULL',
        'table_alias': 'eh',
        'include_address': True
    },
    
    # ============================================================
    # DATA BROKER & ADDITIONAL CONSTRAINTS
    # ============================================================
    {
        'name': 'pnf',
        'source_table': 'PNF_Approvals',
        'join_geom': 'buffered_geom',
        'columns': {'CONTROLLIN': 'pnf_controllin'},
        'where_clause': 'pnf.CONTROLLIN IS NOT NULL',
        'table_alias': 'pnf'
    },
    {
        'name': 'bct',
        'source_table': 'BCT_Agreements',
        'join_geom': 'buffered_geom',
        'columns': {'CONTROLLIN': 'bct_controllin'},
        'where_clause': 'bct.CONTROLLIN IS NOT NULL',
        'table_alias': 'bct'
    },
    {
        'name': 'airportnoise',
        'source_table': 'airport_noise',
        'join_geom': 'buffered_geom',
        'columns': {'anef_code': 'airport_anef_code'},
        'where_clause': 'anef.anef_code IS NOT NULL',
        'table_alias': 'anef'
    },
    {
        'name': 'floodsdf',
        'source_table': 'flood_sfd_1aep',
        'join_geom': 'buffered_geom',
        'columns': {'ogc_fid': 'floodsdf_ogc_fid'},
        'where_clause': 'fsdf.ogc_fid IS NOT NULL',
        'table_alias': 'fsdf',
        'geom_column': 'wkb_geometry'
    },
    {
        'name': 'crownreserves',
        'source_table': 'crown_council_reserves',
        'join_geom': 'buffered_geom',
        'columns': {'reserve_name': 'crown_reserve_name'},
        'where_clause': 'crown.reserve_name IS NOT NULL',
        'table_alias': 'crown'
    },
    {
        'name': 'contamination',
        'source_table': 'contamination_sites',
        'join_geom': 'buffered_geom',
        'columns': {'sitename': 'contamination_sitename'},
        'where_clause': 'contam.sitename IS NOT NULL',
        'table_alias': 'contam'
    },
    {
        'name': 'ramsarwetland',
        'source_table': 'ramsar_wetlands',
        'join_geom': 'buffered_geom',
        'columns': {'source': 'ramsar_source', 'refcode': 'ramsar_refcode'},
        'where_clause': 'ramsar.refcode IS NOT NULL',
        'table_alias': 'ramsar'
    },
    {
        'name': 'specialcontrolledarea',
        'source_table': 'special_and_controlled_areas_wnsw',
        'join_geom': 'buffered_geom',
        'columns': {'class': 'sca_class', 'label': 'sca_label', 'legislation': 'sca_legislation'},
        'where_clause': 'sca.class IS NOT NULL',
        'table_alias': 'sca'
    },
    {
        'name': 'localcomplying',
        'source_table': 'Local_Complying_Exclusion',
        'join_geom': 'buffered_geom',
        'columns': {'LAY_CLASS': 'localcomplying_lay_class'},
        'where_clause': 'lce.LAY_CLASS IS NOT NULL',
        'table_alias': 'lce'
    },
    {
        'name': 'npwsestate',
        'source_table': 'national_parks_and_wildlife_service_estate',
        'join_geom': 'buffered_geom',
        'columns': {'ogc_fid': 'npws_ogc_fid'},
        'where_clause': 'npws.ogc_fid IS NOT NULL',
        'table_alias': 'npws'
    },
    {
        'name': 'localprovision',
        'source_table': 'local_provisions',
        'join_geom': 'buffered_geom',
        'columns': {'lay_name': 'localprov_lay_name', 'lay_class': 'localprov_lay_class'},
        'where_clause': 'lprov.lay_class IS NOT NULL',
        'table_alias': 'lprov'
    },
    {
        'name': 'landreservation',
        'source_table': 'land_reservation_acquisition',
        'join_geom': 'buffered_geom',
        'columns': {'lra_type': 'landres_lra_type', 'lay_class': 'landres_lay_class', 'label': 'landres_label'},
        'where_clause': 'lra.lra_type IS NOT NULL',

    }
    
]

print(f"\n✓ Total: {len(layer_config)} layers configured for CDC compliance")

print(f"Configured {len(layer_config)} layers for processing")
print(f"  - Data Broker & Additional: 12 layers (PNF, BCT, Airport Noise, Flood SDF, Crown Reserves, Contamination, Ramsar, Special Areas, Local Complying, NPWS, Local Provision, Land Reservation)")

print(f"  - Base/Reference: 2 layers")
print(f"  - Heritage & Cultural: 1 layer")

print(f"  - Planning Controls: 11 layers (includes historic zoning and LMR)")
print(f"  - Environmental Constraints: 22 layers")


✓ Total: 49 layers configured for CDC compliance
Configured 49 layers for processing
  - Data Broker & Additional: 12 layers (PNF, BCT, Airport Noise, Flood SDF, Crown Reserves, Contamination, Ramsar, Special Areas, Local Complying, NPWS, Local Provision, Land Reservation)
  - Base/Reference: 2 layers
  - Heritage & Cultural: 1 layer
  - Planning Controls: 11 layers (includes historic zoning and LMR)
  - Environmental Constraints: 22 layers


In [12]:
def process_single_layer(con, layer, base_table='lot_property_landvaluation'):
    """
    Process a single layer: create raw table and aggregated table.
    
    Args:
        con: DuckDB connection
        layer: Layer configuration dictionary
        base_table: Name of the base table to join against
    """
    layer_name = layer['name']
    table_name = f"up_property_{layer_name}"
    aggregated_table_name = f"{table_name}_aggregated"
    
    # Drop existing tables
    con.execute(f"DROP TABLE IF EXISTS {table_name}")
    
    # Build column selections
    base_columns = ['p.gurasid', 'p.lotnumber', 'p.sectionnumber', 'p.planlabel']
    
    # Add special columns if specified
    if layer.get('include_address'):
        base_columns.append('p.address')
    if layer.get('include_propid'):
        base_columns.append('p.propid')
    
    # Add layer-specific columns
    alias = layer['table_alias']
    column_selections = []
    for source_col, target_col in layer['columns'].items():
        column_selections.append(f"{alias}.{source_col} AS {target_col}")
    
    # Determine join condition
    join_type = layer.get('join_type', 'geometry')
    
    if join_type == 'geometry':
        join_geom = layer['join_geom']
        
        # Check if geometry column is explicitly specified, otherwise auto-detect
        if 'geom_column' in layer:
            geom_column = layer['geom_column']
        else:
            # Check if source table has 'geom', 'geometry', or 'wkb_geometry' column
            try:
                col_query = f"SELECT * FROM {PG_ATTACHMENT_NAME}.{PG_SCHEMA}.{layer['source_table']} LIMIT 0"
                result = con.execute(col_query)
                columns = [desc[0].lower() for desc in result.description]
                
                if 'geometry' in columns:
                    geom_column = 'geometry'
                elif 'geom' in columns:
                    geom_column = 'geom'
                elif 'wkb_geometry' in columns:
                    geom_column = 'wkb_geometry'
                else:
                    geom_column = 'geom'
            except:
                geom_column = 'geom'
        
        try_geom = f"TRY_CAST({alias}.{geom_column} AS GEOMETRY)"
        # Optionally extract polygonal components (e.g., MULTISURFACE -> POLYGON/MULTIPOLYGON)
        if layer.get('collection_extract_dim'):
            try_geom = f"ST_CollectionExtract({try_geom}, {layer['collection_extract_dim']})"
        # Convert via WKT to ensure DuckDB parser gets a supported type
        geom_expr = f"ST_GeomFromText(ST_AsText({try_geom}))"
        join_condition = f"ST_Intersects((p.{join_geom}), {geom_expr})"
    elif join_type == 'address':
        join_condition = f"{alias}.address = p.address"
    elif join_type == 'propid':
        join_condition = f"{alias}.propid = p.propid"
        if 'p.propid' not in base_columns:
            base_columns.insert(0, 'p.propid')
    
    # Build and execute the raw table query
    raw_query = f"""
    CREATE TABLE {table_name} AS
    WITH base_data AS (
        SELECT *
        FROM {base_table}
        WHERE centroid_geom IS NOT NULL
    )
    SELECT DISTINCT
        {', '.join(base_columns)},
        {', '.join(column_selections)}
    FROM base_data p
    LEFT JOIN {PG_ATTACHMENT_NAME}.{PG_SCHEMA}.{layer['source_table']} {alias}
        ON {join_condition}
    WHERE {layer['where_clause']}
    """
    
    con.execute(raw_query)
    
    # Drop existing aggregated table
    con.execute(f"DROP TABLE IF EXISTS {aggregated_table_name}")
    
    # Build aggregation query
    agg_method = layer.get('aggregation_method', 'string')
    
    agg_selections = []
    for source_col, target_col in layer['columns'].items():
        if agg_method == 'max_and_string':
            if target_col in ['width', 'depth', 'propertyfrontagecount']:
                agg_selections.append(f"MAX({target_col}) AS {target_col}")
            else:
                agg_selections.append(f"STRING_AGG(DISTINCT {target_col}, ', ') AS {target_col}")
        elif agg_method == 'string_agg':
            date_col = None
            for col_name in layer['columns'].values():
                if 'from_date' in col_name.lower() or 'commenced_date' in col_name.lower():
                    date_col = col_name
                    break
            if 'date' in target_col.lower() or 'datetime' in target_col.lower():
                if date_col:
                    agg_selections.append(f"STRING_AGG(CAST({target_col} AS VARCHAR), ' | ' ORDER BY {date_col}) AS {target_col}")
                else:
                    agg_selections.append(f"STRING_AGG(CAST({target_col} AS VARCHAR), ' | ') AS {target_col}")
            else:
                if date_col:
                    agg_selections.append(f"STRING_AGG({target_col}, ' | ' ORDER BY {date_col}) AS {target_col}")
                else:
                    agg_selections.append(f"STRING_AGG({target_col}, ' | ') AS {target_col}")
        else:
            if any(keyword in target_col.lower() for keyword in ['height', 'fsr', 'size', 'postcode']):
                agg_selections.append(f"STRING_AGG(DISTINCT CAST({target_col} AS VARCHAR), ', ') AS {target_col}")
            else:
                agg_selections.append(f"STRING_AGG(DISTINCT {target_col}, ', ') AS {target_col}")
    
    # Create aggregated table
    agg_query = f"""
    CREATE TABLE {aggregated_table_name} AS
    WITH base AS (
        SELECT DISTINCT
            gurasid,
            lotnumber,
            sectionnumber,
            planlabel,
            {', '.join([target_col for _, target_col in layer['columns'].items()])}
        FROM {table_name}
    )
    SELECT
        gurasid,
        lotnumber,
        sectionnumber,
        planlabel,
        {', '.join(agg_selections)}
    FROM base
    GROUP BY gurasid, lotnumber, sectionnumber, planlabel
    """
    
    con.execute(agg_query)
    
    return table_name, aggregated_table_name

In [13]:
def calculate_poi_distances(con, PG_SCHEMA, PG_ATTACHMENT_NAME, base_table='lot_property_join'):
    """
    Calculate distances to nearest POIs (Hospital, School, Railway Station).
    Creates lot_property_poi table with distance metrics.
    Executes query in PostgreSQL (native KNN support), then loads into DuckDB.
    
    Args:
        con: DuckDB connection
        PG_SCHEMA: PostgreSQL schema name
        PG_ATTACHMENT_NAME: PostgreSQL attachment name in DuckDB
        base_table: Name of the base table to join against
    """
    print("  Calculating POI distances (Hospital, School, Railway Station)...")
    
    # First, export the base table to PostgreSQL temporarily
    print("    Exporting base data to PostgreSQL...")
    
    # IMPORTANT: Deduplicate by lot_section_plan first to avoid duplicate POI records
    # (lot_section_plan = lotnumber/sectionnumber/planlabel is the unique lot identifier)
    base_data = con.execute(f"""
        SELECT 
            objectid,
            planlabel,
            lotnumber,
            sectionnumber,
            ST_AsWKB(centroid_geom) as centroid_geom_wkb,
            CONCAT(lotnumber, '/', COALESCE(sectionnumber, ''), '/', planlabel) AS lot_section_plan
        FROM (
            SELECT DISTINCT ON (lotnumber, sectionnumber, planlabel) *
            FROM {base_table}
            ORDER BY lotnumber, sectionnumber, planlabel
        ) unique_base
    """).fetchdf()
    
    unique_count = len(base_data)
    print(f"    Base data exported: {unique_count} unique lot_section_plans")
    
    # Use SQLAlchemy to write to PostgreSQL
    from sqlalchemy import create_engine, text
    pg_engine = create_engine(POSTGRES_URL_SQLALCHEMY)
    
    # Drop temp table if exists
    with pg_engine.connect() as conn:
        conn.execute(text(f"DROP TABLE IF EXISTS {PG_SCHEMA}.temp_lot_property_base"))
        conn.commit()
    
    # Export to PostgreSQL
    base_data_export = base_data[['objectid', 'planlabel', 'lotnumber', 'sectionnumber', 'lot_section_plan', 'centroid_geom_wkb']].copy()
    base_data_export.to_sql(
        name='temp_lot_property_base',
        con=pg_engine,
        schema=PG_SCHEMA,
        if_exists='replace',
        index=False,
        method='multi'
    )
    
    print("    Calculating POI distances in PostgreSQL...")
    
    # Execute POI calculation in PostgreSQL using native KNN operator
    # Transform from SRID 4283 to 4326 to match POI table
    poi_query = f"""
    CREATE TABLE {PG_SCHEMA}.temp_lot_property_poi AS
    WITH base_with_geom AS (
        SELECT 
            objectid,
            planlabel,
            lotnumber,
            sectionnumber,
            lot_section_plan,
            ST_Transform(ST_SetSRID(ST_GeomFromWKB(centroid_geom_wkb::bytea), 4283), 4326) as centroid_geom
        FROM {PG_SCHEMA}.temp_lot_property_base
    )
    SELECT 
        prop.objectid,
        prop.planlabel,
        prop.lotnumber,
        prop.sectionnumber,
        prop.lot_section_plan,
        
        hospital.poiname AS closest_hospital,
        ROUND(ST_Distance(prop.centroid_geom::geography, hospital.geometry::geography)) AS closest_hospital_distance_m,
        
        school.poiname AS closest_school,
        ROUND(ST_Distance(prop.centroid_geom::geography, school.geometry::geography)) AS closest_school_distance_m,
        
        railway.poiname AS closest_railway_station,
        ROUND(ST_Distance(prop.centroid_geom::geography, railway.geometry::geography)) AS closest_railway_station_distance_m
    
    FROM base_with_geom prop
    
    CROSS JOIN LATERAL (
        SELECT geometry, poiname
        FROM {PG_SCHEMA}.points_of_interest
        WHERE poigroup = 1
        ORDER BY prop.centroid_geom <-> geometry
        LIMIT 1
    ) hospital
    
    CROSS JOIN LATERAL (
        SELECT geometry, poiname
        FROM {PG_SCHEMA}.points_of_interest
        WHERE poigroup = 2
        ORDER BY prop.centroid_geom <-> geometry
        LIMIT 1
    ) school
    
    CROSS JOIN LATERAL (
        SELECT geometry, poiname
        FROM {PG_SCHEMA}.points_of_interest
        WHERE poigroup = 4
        ORDER BY prop.centroid_geom <-> geometry
        LIMIT 1
    ) railway
    """
    
    with pg_engine.connect() as conn:
        # Drop temp POI table if exists
        conn.execute(text(f"DROP TABLE IF EXISTS {PG_SCHEMA}.temp_lot_property_poi"))
        conn.commit()
        
        # Create POI distance table in PostgreSQL
        conn.execute(text(poi_query))
        conn.commit()
    
    print("    Loading POI results back into DuckDB...")
    
    # Read results back into DuckDB
    con.execute("DROP TABLE IF EXISTS lot_property_poi")
    con.execute(f"""
        CREATE TABLE lot_property_poi AS
        SELECT * FROM {PG_ATTACHMENT_NAME}.{PG_SCHEMA}.temp_lot_property_poi
    """)
    
    # Clean up temporary PostgreSQL tables
    with pg_engine.connect() as conn:
        conn.execute(text(f"DROP TABLE IF EXISTS {PG_SCHEMA}.temp_lot_property_base"))
        conn.execute(text(f"DROP TABLE IF EXISTS {PG_SCHEMA}.temp_lot_property_poi"))
        conn.commit()
    
    count = con.execute("SELECT COUNT(*) FROM lot_property_poi").fetchone()[0]
    print(f"    ✓ Created lot_property_poi with {count} rows")


# Helper Functions

These functions enable configuration-driven layer processing:

1. **`process_single_layer()`** - Processes a single layer configuration:
   - Creates raw table with spatial/attribute joins
   - Creates aggregated table with STRING_AGG
   - Handles different join types (geometry, address, propid)
   - Manages special aggregation methods

2. **`add_lot_section_plan_columns()`** - Adds computed lot_section_plan column to all aggregated tables

These functions are called by `run_notebook()` to process all layers defined in `layer_config`.

In [14]:
def add_lot_section_plan_columns(con):
    """Add lot_section_plan column to all aggregated tables if missing."""
    tables_query = """
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_name LIKE 'up_property_%_aggregated'
    """
    aggregated_tables = con.execute(tables_query).fetchall()

    for table in aggregated_tables:
        table_name = table[0]
        column_query = f"""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = '{table_name}' AND column_name = 'lot_section_plan'
        """
        column_exists = con.execute(column_query).fetchone()

        if not column_exists:
            try:
                con.execute(f"""
                ALTER TABLE {table_name}
                ADD COLUMN lot_section_plan VARCHAR;

                UPDATE {table_name}
                SET lot_section_plan = CONCAT(lotnumber, '/', COALESCE(sectionnumber, ''), '/', planlabel);
                """)
                count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
                print(f"    Added lot_section_plan to {table_name} ({count} rows)")
            except Exception as e:
                print(f"    Error updating {table_name}: {e}")


def run_notebook():
    """
    Main function: Process all councils and enrich property data with planning layers.
    Uses layer_config defined above for configuration-driven layer processing.
    """
    
    # Fetch all unique council names
    councils = con.execute(f"""
        SELECT DISTINCT council FROM {PG_ATTACHMENT_NAME}.{PG_SCHEMA}."addressstring"  where council ilike '%LIVERPOOL%'
    """).fetchdf()

    for council_name in councils['council']:
        print(f"\n{'='*70}")
        print(f"PROCESSING COUNCIL: {council_name}")
        print(f"{'='*70}\n")

        # ============================================================
        # STEP 1: Create Property Address Summary
        # ============================================================
        print("Step 1: Creating property address summary...")
        con.execute("DROP TABLE IF EXISTS property_address_summary")

        con.execute(f"""
        CREATE TABLE property_address_summary AS
        SELECT 
            p.PROPID as propid, 
            a.address,
            a.gurasid,
            a.postcode,
            a.council,
            p.geometry,
            COUNT(*) OVER (PARTITION BY p.PROPID) AS propid_count
        FROM {PG_ATTACHMENT_NAME}.{PG_SCHEMA}."Property" p
        INNER JOIN {PG_ATTACHMENT_NAME}.{PG_SCHEMA}."addressstring" a
            ON p.PROPID =a.propid
        WHERE a.council = '{council_name}'
        """)
        
        # ============================================================
        # STEP 2: Create Lot Geometries (Centroid + Buffered)
        # ============================================================
        print("Step 2: Creating lot geometries (centroid + buffered)...")
        con.execute("DROP TABLE IF EXISTS lot_geometries_4283")

        con.execute(f"""
        CREATE TABLE lot_geometries_4283 AS
        SELECT 
            OBJECTID as objectid,
            geometry,
            PLANLABEL as planlabel,
            LOTNUMBER as lotnumber,
            SECTIONNUMBER as sectionnumber,
            AREA_H as area_h,
            ST_Centroid(ST_MakeValid(ST_GeomFromWKB(geometry))) AS centroid_geom,
            ST_Buffer(ST_GeomFromWKB(geometry), -0.000009) AS buffered_geom
        FROM {PG_ATTACHMENT_NAME}.{PG_SCHEMA}."Lot"
        WHERE geometry IS NOT NULL
        """)

        # ============================================================
        # STEP 3: Join Lots with Properties (using buffered geometry)
        # ============================================================
        print("Step 3: Joining lots with properties (using buffered geometry)...")
        con.execute("DROP TABLE IF EXISTS lot_property_join")
        
        con.execute("""
        CREATE TABLE lot_property_join AS
        SELECT 
            lg.objectid,
            lg.planlabel,
            lg.lotnumber,
            lg.sectionnumber,
            lg.area_h,
            lg.centroid_geom,
            lg.buffered_geom,
            lg.geometry,
            pas.postcode,
            pas.propid,
            pas.address,
            pas.gurasid,
            pas.propid_count
        FROM lot_geometries_4283 lg
        JOIN property_address_summary pas
            ON ST_Intersects(lg.buffered_geom, CAST(pas.geometry AS GEOMETRY))
        """)

        # ============================================================
        # STEP 3.5: Calculate POI Distances
        # ============================================================
        print("Step 3.5: Calculating POI distances...")
        calculate_poi_distances(con, PG_SCHEMA, PG_ATTACHMENT_NAME)
        print()

        # ============================================================
        # STEP 4: Create Base Table with Land Valuation
        # ============================================================
        print("Step 4: Creating base table with land valuation...")
        con.execute("DROP TABLE IF EXISTS lot_property_landvaluation")

        con.execute(f"""
        CREATE TABLE lot_property_landvaluation AS
        SELECT 
            lpj.objectid,
            lpj.planlabel,
            lpj.lotnumber,
            lpj.sectionnumber,
            lpj.postcode,
            lpj.propid,
            lpj.address,
            lpj.gurasid,
            lpj.propid_count,
            CONCAT(lpj.lotnumber, '/', COALESCE(lpj.sectionnumber, ''), '/', lpj.planlabel) AS lot_section_plan,
            lpj.centroid_geom,
            lpj.buffered_geom,
            lpj.area_h,
            lv.property_description,
            lv.land_value_1
        FROM lot_property_join lpj
        LEFT JOIN {PG_ATTACHMENT_NAME}.{PG_SCHEMA}."land_valuation" lv
            ON lpj.propid = lv.property_id
        """)
        
        count = con.execute("SELECT COUNT(*) FROM lot_property_landvaluation").fetchone()[0]
        print(f"   Created base table with {count} rows\n")
        
        # ============================================================
        # STEP 4.5: Join POI Distances
        # ============================================================
        print("Step 4.5: Joining POI distances...")
        
        # POI table is already deduplicated at source (based on lot_section_plan)
        # Join on lot_section_plan to match unique lots
        con.execute(f"""
        CREATE TABLE lot_property_landvaluation_with_poi AS
        SELECT 
            lpv.*,
            poi.closest_hospital,
            poi.closest_hospital_distance_m,
            poi.closest_school,
            poi.closest_school_distance_m,
            poi.closest_railway_station,
            poi.closest_railway_station_distance_m
        FROM lot_property_landvaluation lpv
        LEFT JOIN lot_property_poi poi
            ON lpv.lot_section_plan = poi.lot_section_plan
        """)
        
        # Replace old table with new one
        con.execute("DROP TABLE lot_property_landvaluation")
        con.execute("ALTER TABLE lot_property_landvaluation_with_poi RENAME TO lot_property_landvaluation")
        
        count = con.execute("SELECT COUNT(*) FROM lot_property_landvaluation").fetchone()[0]
        print(f"   ✓ Enriched base table with POI data ({count} rows)\n")
        
        # ============================================================
        # STEP 5: Process All Configured Layers
        # ============================================================
        print(f"Step 5: Processing {len(layer_config)} planning layers...")
        print(f"{'='*70}")
        
        for i, layer in enumerate(layer_config, 1):
            layer_name = layer['name']
            print(f"  [{i}/{len(layer_config)}] Processing: {layer_name}")
            
            try:
                process_single_layer(con, layer)
            except Exception as e:
                print(f"      ERROR processing {layer_name}: {e}")
                continue
        
        print(f"{'='*70}")
        print(f"Completed processing all layers\n")
        
        # ============================================================
        # STEP 6: Add lot_section_plan Columns
        # ============================================================
        print("Step 6: Adding lot_section_plan columns to aggregated tables...")
        add_lot_section_plan_columns(con)
        print()

        # ============================================================
        # STEP 7: Create Comprehensive Table
        # ============================================================
        print("Step 7: Creating comprehensive table with all layers...")
        con.execute("DROP TABLE IF EXISTS up_property_comprehensive")
        
        # Get list of available aggregated tables
        available_tables = con.execute("""
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_name LIKE 'up_property_%_aggregated'
        """).fetchdf()['table_name'].tolist()
        
        print(f"   Found {len(available_tables)} aggregated tables to join")
        
        # Build dynamic JOIN query based on available tables
        select_parts = ['base.*']
        join_parts = []
        
        # Map of expected table suffixes to their aliases
        table_map = {
            'suburb': 'suburb',
            'lga': 'lga',
            'lep': 'lep',
            'dcp': 'dcp',
            'lzn': 'lzn',
            'lzn_historic': 'lznh',
            'fsr': 'fsr',
            'hob': 'hob',
            'minlotsize': 'mls',
            'activestreetfront': 'asf',
            'lmr': 'lmr',
            'foreshorebuildline': 'fbl',
            'greenfieldhousecode': 'ghc',
            'flood': 'flood',
            'bushfire': 'bush',
            'acidsulfatesoils': 'acid',
            'biodiversity': 'bio',
            'groundwater': 'gw',
            'landslide': 'landslide',
            'biodiversitymap': 'biomap',
            'biodiversityvalues': 'bioval',
            'envsensitiveland': 'envsensi',
            'wilderness': 'wild',
            'mineralresource': 'mineral',
            'natresbiodiversity': 'nrbio',
            'natressensitivity': 'nrsensi',
            'natreswater': 'nrwater',
            'salinity': 'sal',
            'koalahabitat': 'koala',
            'coastalwetlands': 'cwet',
            'coastalenvironment': 'cenv',
            'coastaluse': 'cuse',
            'drinkingwatercatch': 'dwc',
            'riparianland': 'rip',
            'minesubsidence': 'msub',
            'scenicprotection': 'scenic',
            'heritage': 'heritage',
            'pnf': 'pnf',
            'bct': 'bct',
            'airportnoise': 'anef',
            'floodsdf': 'fsdf',
            'crownreserves': 'crown',
            'contamination': 'contam',
            'ramsarwetland': 'ramsar',
            'specialcontrolledarea': 'sca',
            'localcomplying': 'lce',
            'npwsestate': 'npws',
            'localprovision': 'lprov',
            'landreservation': 'lra'
        }
        
        for suffix, alias in table_map.items():
            table_name = f'up_property_{suffix}_aggregated'
            if table_name in available_tables:
                select_parts.append(f"{alias}.* EXCLUDE (gurasid, lotnumber, sectionnumber, planlabel, lot_section_plan)")
                join_parts.append(f"LEFT JOIN {table_name} {alias} ON base.gurasid = {alias}.gurasid AND base.lot_section_plan = {alias}.lot_section_plan")
        
        # Build and execute comprehensive query
        comprehensive_query = f"""
        CREATE TABLE up_property_comprehensive AS
        SELECT 
            {', '.join(select_parts)}
        FROM lot_property_landvaluation base
        {' '.join(join_parts)}
        WHERE base.centroid_geom IS NOT NULL
        """
        
        con.execute(comprehensive_query)

        count = con.execute("SELECT COUNT(*) FROM up_property_comprehensive").fetchone()[0]
        print(f"   Created comprehensive table with {count} rows\n")

        # ============================================================
        # STEP 7.5: Enrich with Permissible Land Uses (per-batch)
        # ============================================================
        print("Step 7.5: Enriching with permissible land uses...")

        con.execute("DROP TABLE IF EXISTS up_property_with_permissible_uses")

        con.execute(f"""
        CREATE TABLE up_property_with_permissible_uses AS
        WITH zone_splits AS (
            -- Split comma-separated zones into individual rows
            SELECT 
                objectid,
                gurasid,
                lot_section_plan,
                lep_name,
                TRIM(UNNEST(string_split(lzn_sym_code, ','))) AS zone_code
            FROM up_property_comprehensive
            WHERE lzn_sym_code IS NOT NULL 
                AND lep_name IS NOT NULL
        ),
        matched_uses AS (
            -- Match each zone with permissible uses
            SELECT 
                zs.objectid,
                zs.gurasid,
                zs.lot_section_plan,
                plu.permissiblelanduse
            FROM zone_splits zs
            INNER JOIN {PG_ATTACHMENT_NAME}.{PG_SCHEMA}.up_permissiblelanduse plu
                ON zs.lep_name = plu.epititle
                AND zs.zone_code = plu.zone
            WHERE plu.permissiblelanduse IS NOT NULL
        ),
        aggregated_uses AS (
            -- Aggregate all permissible uses per property
            SELECT 
                objectid,
                gurasid,
                lot_section_plan,
                STRING_AGG(DISTINCT permissiblelanduse, ', ') AS permissible_uses
            FROM matched_uses
            GROUP BY objectid, gurasid, lot_section_plan
        )
        -- Join back to comprehensive table
        SELECT 
            comp.*,
            agg.permissible_uses
        FROM up_property_comprehensive comp
        LEFT JOIN aggregated_uses agg
            ON comp.objectid = agg.objectid
            AND comp.gurasid = agg.gurasid
            AND comp.lot_section_plan = agg.lot_section_plan
        """)

        con.execute("DROP TABLE up_property_comprehensive")
        con.execute("ALTER TABLE up_property_with_permissible_uses RENAME TO up_property_comprehensive")

        pu_count = con.execute("SELECT COUNT(*) FROM up_property_comprehensive").fetchone()[0]
        with_uses = con.execute("SELECT COUNT(*) FROM up_property_comprehensive WHERE permissible_uses IS NOT NULL").fetchone()[0]
        print(f"   ✓ Enriched {pu_count} properties, {with_uses} have permissible uses\n")

        # ============================================================
        # STEP 7.6: Enrich with Lot Metrics (per-batch)
        # ============================================================
        print("Step 7.6: Enriching with lot metrics...")

        con.execute("DROP TABLE IF EXISTS up_property_with_lot_metrics")

        con.execute(f"""
        CREATE TABLE up_property_with_lot_metrics AS
        SELECT 
            comp.*,
            lm.area_sqm,
            lm.perimeter_m,
            lm.longest_axis_m,
            lm.orientation_degrees,
            lm.elongation,
            lm.rectangularity,
            lm.shape_index,
            lm.circular_compactness,
            lm.convexity,
            lm.fractal_dimension,
            lm.equivalent_rectangular_index,
            lm.square_compactness,
            lm.corners_count,
            lm.centroid_lat,
            lm.centroid_lon,
            lm.min_width_m,
            lm.effective_diameter_m,
            lm.neck_ratio,
            lm.num_frontages,
            lm.is_corner_lot,
            lm.primary_frontage_road,
            lm.primary_frontage_length_m,
            lm.all_frontages,
            lm.all_frontage_road_ids,
            lm.all_edges_measurements,
            lm.frontage_area_ratio,
            lm.is_battleaxe
        FROM up_property_comprehensive comp
        LEFT JOIN {PG_ATTACHMENT_NAME}.{PG_SCHEMA}.lot_metrics_3 lm
            ON comp.lot_section_plan = CONCAT(lm.lotnumber, '/', COALESCE(lm.sectionnumber, ''), '/', lm.planlabel)
        """)

        con.execute("DROP TABLE up_property_comprehensive")
        con.execute("ALTER TABLE up_property_with_lot_metrics RENAME TO up_property_comprehensive")

        lm_count = con.execute("SELECT COUNT(*) FROM up_property_comprehensive WHERE area_sqm IS NOT NULL").fetchone()[0]
        print(f"   ✓ Enriched with lot metrics, {lm_count} properties have metrics\n")

        # ============================================================
        # STEP 7.7: Enrich with Slope Data (per-batch)
        # ============================================================
        print("Step 7.7: Enriching with slope data...")

        con.execute("DROP TABLE IF EXISTS up_property_with_slope")

        con.execute(f"""
        CREATE TABLE up_property_with_slope AS
        SELECT 
            comp.*,
            lws.average_slope
        FROM up_property_comprehensive comp
        LEFT JOIN {PG_ATTACHMENT_NAME}.{PG_SCHEMA}.lotwithslope lws
            ON comp.lot_section_plan = lws.lot_section_plan
        """)

        con.execute("DROP TABLE up_property_comprehensive")
        con.execute("ALTER TABLE up_property_with_slope RENAME TO up_property_comprehensive")

        slope_count = con.execute("SELECT COUNT(*) FROM up_property_comprehensive WHERE average_slope IS NOT NULL").fetchone()[0]
        print(f"   ✓ Enriched with slope data, {slope_count} properties have slope\n")

        # ============================================================
        # STEP 8: Export to PostgreSQL
        # ============================================================
        print("Step 8: Exporting to PostgreSQL...")
        
        # Convert geometry columns to WKT format for easier PostgreSQL import
        data_to_export = con.execute("""
            SELECT 
                * EXCLUDE (centroid_geom, buffered_geom),
                ST_AsText(centroid_geom) as centroid_geom_wkt,
                ST_AsText(buffered_geom) as buffered_geom_wkt
            FROM up_property_comprehensive
        """).fetchdf()
        
        print(f"   Exporting {data_to_export.shape[0]} rows and {data_to_export.shape[1]} columns")
        engine = create_engine(POSTGRES_URL_SQLALCHEMY)

        data_to_export.to_sql(
            name="up_property_comprehensive",
            con=engine,
            schema=PG_SCHEMA,
            if_exists="append",
            index=False
        )

        # Verify export
        with engine.connect() as conn:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {PG_SCHEMA}.up_property_comprehensive"))
            postgres_count = result.fetchone()[0]
            print(f"   Successfully exported {postgres_count} rows to PostgreSQL")

    print("\n" + "="*70)
    print("ALL COUNCILS PROCESSED SUCCESSFULLY!")
    print("="*70)


In [15]:
con.execute("DROP TABLE IF EXISTS up_property_comprehensive")

In [16]:
with engine.connect() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {PG_SCHEMA}.up_property_comprehensive"))
    conn.commit()

In [ ]:

# Example usage of the run_notebook function
if __name__ == "__main__":
    run_notebook()



PROCESSING COUNCIL: LIVERPOOL

Step 1: Creating property address summary...
Step 2: Creating lot geometries (centroid + buffered)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Step 3: Joining lots with properties (using buffered geometry)...
Step 3.5: Calculating POI distances...
  Calculating POI distances (Hospital, School, Railway Station)...
    Exporting base data to PostgreSQL...
    Base data exported: 70248 unique lot_section_plans
    Calculating POI distances in PostgreSQL...
    Loading POI results back into DuckDB...
    ✓ Created lot_property_poi with 70248 rows

Step 4: Creating base table with land valuation...
   Created base table with 97748 rows

Step 4.5: Joining POI distances...
   ✓ Enriched base table with POI data (97748 rows)

Step 5: Processing 49 planning layers...
  [1/49] Processing: suburb


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  [2/49] Processing: lga


...

  [18/49] Processing: groundwater
  [19/49] Processing: landslide
  [20/49] Processing: biodiversitymap
  [21/49] Processing: biodiversityvalues
  [22/49] Processing: envsensitiveland
  [23/49] Processing: wilderness
  [24/49] Processing: mineralresource
  [25/49] Processing: natresbiodiversity
  [26/49] Processing: natressensitivity
  [27/49] Processing: natreswater
  [28/49] Processing: salinity
  [29/49] Processing: koalahabitat
  [30/49] Processing: coastalwetlands
  [31/49] Processing: coastalenvironment
  [32/49] Processing: coastaluse
  [33/49] Processing: drinkingwatercatch
  [34/49] Processing: riparianland
  [35/49] Processing: minesubsidence
  [36/49] Processing: scenicprotection
  [37/49] Processing: heritage
  [38/49] Processing: pnf
  [39/49] Processing: bct
  [40/49] Processing: airportnoise
  [41/49] Processing: floodsdf


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  [42/49] Processing: crownreserves
  [43/49] Processing: contamination
  [44/49] Processing: ramsarwetland
  [45/49] Processing: specialcontrolledarea
  [46/49] Processing: localcomplying
  [47/49] Processing: npwsestate
  [48/49] Processing: localprovision
  [49/49] Processing: landreservation
      ERROR processing landreservation: 'table_alias'
Completed processing all layers

Step 6: Adding lot_section_plan columns to aggregated tables...
    Added lot_section_plan to up_property_acidsulfatesoils_aggregated (2 rows)
    Added lot_section_plan to up_property_activestreetfront_aggregated (0 rows)
    Added lot_section_plan to up_property_airportnoise_aggregated (0 rows)
    Added lot_section_plan to up_property_bct_aggregated (109 rows)
    Added lot_section_plan to up_property_biodiversitymap_aggregated (0 rows)
    Added lot_section_plan to up_property_biodiversityvalues_aggregated (97 rows)
    Added lot_section_plan to up_property_biodiversity_aggregated (42 rows)
    Added lot_

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ✓ Enriched with lot metrics, 11828 properties have metrics

Step 7.7: Enriching with slope data...
   ✓ Enriched with slope data, 14639 properties have slope

Step 8: Exporting to PostgreSQL...
   Exporting 14754 rows and 169 columns
   Successfully exported 112504 rows to PostgreSQL

ALL COUNCILS PROCESSED SUCCESSFULLY!


In [ ]:
# ============================================================
# NOTE: Lot metrics enrichment has been moved into
# run_notebook() as Step 7.6. It now runs per-council batch
# in DuckDB before the PostgreSQL export via a LEFT JOIN
# against lot_metrics_3 on lot_section_plan.
# ============================================================
print("Lot metrics enrichment is now integrated into run_notebook() as Step 7.6.")
print("Re-run run_notebook() to process lot metrics per council batch.")


Lot metrics enrichment is now integrated into run_notebook() as Step 7.6.
Re-run run_notebook() to process lot metrics per council batch.


In [ ]:
# ============================================================
# NOTE: Slope data enrichment has been moved into
# run_notebook() as Step 7.7. It now runs per-council batch
# in DuckDB before the PostgreSQL export via a LEFT JOIN
# against lotwithslope on lot_section_plan.
# ============================================================
print("Slope enrichment is now integrated into run_notebook() as Step 7.7.")
print("Re-run run_notebook() to process slope data per council batch.")


Slope enrichment is now integrated into run_notebook() as Step 7.7.
Re-run run_notebook() to process slope data per council batch.


In [ ]:
# Convert geometry columns from WKT text to PostGIS geometry type
with engine.connect() as conn:
    # Drop existing geometry columns if they exist (in case of re-run)
    conn.execute(text(f"""
        ALTER TABLE {PG_SCHEMA}.up_property_comprehensive
        DROP COLUMN IF EXISTS centroid_geom,
        DROP COLUMN IF EXISTS buffered_geom;
    """))
    
    # Convert centroid_geom from WKT to geometry
    conn.execute(text(f"""
        ALTER TABLE {PG_SCHEMA}.up_property_comprehensive
        ADD COLUMN centroid_geom geometry(Geometry, 4283);
        
        UPDATE {PG_SCHEMA}.up_property_comprehensive
        SET centroid_geom = ST_SetSRID(ST_GeomFromText(centroid_geom_wkt), 4283)
        WHERE centroid_geom_wkt IS NOT NULL AND centroid_geom_wkt != '';
        
        ALTER TABLE {PG_SCHEMA}.up_property_comprehensive
        DROP COLUMN centroid_geom_wkt;
    """))
    
    # Convert buffered_geom from WKT to geometry
    conn.execute(text(f"""
        ALTER TABLE {PG_SCHEMA}.up_property_comprehensive
        ADD COLUMN buffered_geom geometry(Geometry, 4283);
        
        UPDATE {PG_SCHEMA}.up_property_comprehensive
        SET buffered_geom = ST_SetSRID(ST_GeomFromText(buffered_geom_wkt), 4283)
        WHERE buffered_geom_wkt IS NOT NULL AND buffered_geom_wkt != '';
        
        ALTER TABLE {PG_SCHEMA}.up_property_comprehensive
        DROP COLUMN buffered_geom_wkt;
    """))
    
    conn.commit()

print(f"   Geometry columns converted successfully")
print(f"\n{'='*70}")
print(f"{'='*70}\n")

   Geometry columns converted successfully




In [ ]:
# - Data broker layers (PNF and BCT)
# airport noise
# Flood - SDF
# crown council 
# Contamination sites
# ramsar wetland
# Special and Controlled Areas (WNSW)
# Local Complying Exclusion
# national_parks_and_wildlife_service_estate
# Local Provision
# Land Reservation acquisition



# to be added: 1. avm,

<!-- To be added: Poi, avm, do_properties, slope -->